# 🐼 Clean Code - Session 2 (VERSION FORMATEUR)
## Audit et analyse propre avec Pandas

**⚠️ Ce notebook contient les SOLUTIONS - Ne pas distribuer aux étudiants**

**Objectifs de la session :**
- Réaliser un audit complet de données
- Gérer les valeurs manquantes
- Maîtriser la sélection et le filtrage avancés
- Utiliser les expressions régulières
- Présenter proprement les résultats

---

In [ ]:
# Imports
import pandas as pd
import numpy as np
import re

# Configuration de l'affichage
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

## 1. 📥 Chargement du dataset

Nous travaillons sur un dataset de campagne marketing e-commerce avec des problèmes de qualité intentionnels.

In [ ]:
# Chargement des données
df = pd.read_csv('../data/campagne_marketing_ecommerce.csv')

print(f"Dataset chargé : {df.shape[0]} lignes, {df.shape[1]} colonnes")

---
## 2. 🔍 Audit initial

### 2.1 Vue d'ensemble

In [ ]:
# SOLUTION : Afficher les informations générales du DataFrame
df.info()

In [ ]:
# SOLUTION : Afficher les premières lignes
df.head(10)

In [ ]:
# SOLUTION : Afficher les statistiques descriptives
df.describe(include='all')

### 2.2 Analyse des types de données

In [ ]:
# SOLUTION : Afficher les types de chaque colonne
df.dtypes

In [ ]:
# SOLUTION : Colonnes à convertir

# Colonnes à convertir :
# - date_transaction devrait être de type datetime
# - montant_ttc devrait être de type float (actuellement object à cause des formats variés)
# - nb_visites_avant_achat devrait être de type int (mais contient des NaN donc float64)
# - score_satisfaction devrait être de type float (contient des formats variés)
# - canal_acquisition a des valeurs en casse variable (Email, email, EMAIL)

### 2.3 Analyse des valeurs manquantes

In [ ]:
# SOLUTION : Calculer le nombre de valeurs manquantes par colonne
missing = df.isna().sum()
print(missing)

In [ ]:
# SOLUTION : Calculer le pourcentage de valeurs manquantes
missing_pct = (df.isna().sum() / len(df) * 100).round(2)
print(missing_pct.sort_values(ascending=False))

In [ ]:
# SOLUTION : Créer un rapport d'audit des valeurs manquantes

def audit_missing_values(dataframe):
    """
    Génère un rapport sur les valeurs manquantes d'un DataFrame.
    
    Parameters
    ----------
    dataframe : pd.DataFrame
        DataFrame à auditer.
    
    Returns
    -------
    pd.DataFrame
        Rapport avec colonnes : missing_count, missing_pct, dtype.
    """
    report = pd.DataFrame({
        'missing_count': dataframe.isna().sum(),
        'missing_pct': (dataframe.isna().sum() / len(dataframe) * 100).round(2),
        'dtype': dataframe.dtypes
    })
    return report.sort_values('missing_pct', ascending=False)

# Test de la fonction
audit_missing_values(df)

### 2.4 Détection des doublons

In [ ]:
# SOLUTION : Compter les doublons exacts
n_duplicates = df.duplicated().sum()
print(f"Nombre de lignes dupliquées : {n_duplicates}")

In [ ]:
# SOLUTION : Trouver les doublons basés sur certaines colonnes
cols_key = ['client_email', 'date_transaction', 'produit']
n_duplicates_key = df.duplicated(subset=cols_key).sum()
print(f"Doublons potentiels (email+date+produit) : {n_duplicates_key}")

In [ ]:
# SOLUTION : Afficher quelques exemples de doublons
duplicates_mask = df.duplicated(subset=cols_key, keep=False)
df[duplicates_mask].sort_values(cols_key).head(10)

---
## 3. 🧹 Gestion des valeurs manquantes

### 3.1 Analyse par colonne

In [ ]:
# SOLUTION : Analyse de la colonne 'nb_visites_avant_achat'
print("Statistiques de nb_visites_avant_achat :")
print(df['nb_visites_avant_achat'].describe())

In [ ]:
# SOLUTION : Imputer les valeurs manquantes avec la médiane
median_visites = df['nb_visites_avant_achat'].median()

df['nb_visites_avant_achat_clean'] = df['nb_visites_avant_achat'].fillna(median_visites)

print(f"Médiane utilisée : {median_visites}")
print(f"Valeurs manquantes après imputation : {df['nb_visites_avant_achat_clean'].isna().sum()}")

In [ ]:
# SOLUTION : Créer un flag pour les valeurs qui étaient manquantes
df['visites_was_missing'] = df['nb_visites_avant_achat'].isna()

print(f"Nombre de valeurs imputées : {df['visites_was_missing'].sum()}")

### 3.2 Gestion des valeurs "fausses nulles"

In [ ]:
# Identifier les fausses valeurs nulles dans 'score_satisfaction'
print(df['score_satisfaction'].value_counts(dropna=False).head(15))

In [ ]:
# SOLUTION : Remplacer les fausses nulles par de vraies NaN
false_nulls = ['', 'N/A', 'n/a', 'NA', 'null', 'None']

df['score_satisfaction_clean'] = df['score_satisfaction'].replace(false_nulls, np.nan)

# Vérification
print(df['score_satisfaction_clean'].value_counts(dropna=False).head(10))

In [ ]:
# SOLUTION : Convertir en numérique (gérer les valeurs comme '4.5' et '3,5')

df['score_satisfaction_clean'] = (
    df['score_satisfaction_clean']
    .astype(str)  # Convertir en string d'abord
    .str.replace(',', '.')  # Remplacer virgule par point
    .replace('nan', np.nan)  # Remettre les NaN
    .astype(float)  # Convertir en numérique
)

print(df['score_satisfaction_clean'].describe())

---
## 4. 🎯 Sélection et filtrage avancés

### 4.1 Filtrage avec `.loc[]`

In [ ]:
# Nettoyage de la colonne montant

def clean_amount(value):
    """
    Nettoie et convertit une valeur de montant en float.
    Gère les formats : '123.45', '123,45', '123€', 'EUR 123', etc.
    """
    if pd.isna(value) or value in ['', 'N/A']:
        return np.nan
    
    # Convertir en string et nettoyer
    value = str(value)
    value = value.replace('€', '').replace('EUR', '').replace(' ', '')
    value = value.replace(',', '.')
    
    try:
        return float(value)
    except ValueError:
        return np.nan

df['montant_clean'] = df['montant_ttc'].apply(clean_amount)
print(df['montant_clean'].describe())

In [ ]:
# SOLUTION : Filtrer avec .loc[]
# Transactions > 500€ ET catégorie 'Informatique'

high_value_info = df.loc[
    (df['montant_clean'] > 500) & (df['categorie'] == 'Informatique'),
    ['id_transaction', 'produit', 'montant_clean', 'categorie']
]

print(f"Nombre de transactions : {len(high_value_info)}")
high_value_info.head()

### 4.2 Filtrage avec `.query()`

In [ ]:
# SOLUTION : Même filtre avec .query()

result = df.query("montant_clean > 500 and categorie == 'Informatique'")
print(f"Résultat : {len(result)} lignes")

In [ ]:
# SOLUTION : Utiliser une variable dans query()
min_amount = 1000
categories = ['Informatique', 'Téléphonie']

result = df.query("montant_clean > @min_amount and categorie in @categories")
print(f"Transactions > {min_amount}€ en {categories} : {len(result)}")

### 4.3 Chaînage de méthodes (Method Chaining)

In [ ]:
# SOLUTION : Pipeline complet avec chaînage

result = (
    df
    .query("montant_clean > 100")
    .groupby('categorie')
    .agg({'montant_clean': ['sum', 'mean']})
    .sort_values(('montant_clean', 'sum'), ascending=False)
)

result

---
## 5. 🔤 Expressions régulières avec Pandas

### 5.1 Nettoyage des numéros de téléphone

In [ ]:
# Analyse des formats existants
print("Exemples de numéros de téléphone :")
print(df['client_telephone'].dropna().sample(10).values)

In [ ]:
# SOLUTION : Nettoyer les numéros (garder uniquement les chiffres)
# \D signifie "tout ce qui n'est pas un chiffre"

df['telephone_clean'] = (
    df['client_telephone']
    .str.replace(r'\D', '', regex=True)  # Supprimer tout ce qui n'est pas un chiffre
)

print(df['telephone_clean'].dropna().sample(10).values)

In [ ]:
# SOLUTION : Identifier les numéros invalides (pas 10 chiffres)
# ^\d{10}$ signifie : début de chaîne + exactement 10 chiffres + fin de chaîne

valid_phone_mask = df['telephone_clean'].str.match(r'^\d{10}$', na=False)

print(f"Téléphones valides : {valid_phone_mask.sum()}")
print(f"Téléphones invalides : {(~valid_phone_mask & df['telephone_clean'].notna()).sum()}")

# Voir des exemples de téléphones invalides
print("\nExemples de téléphones invalides :")
print(df.loc[~valid_phone_mask & df['telephone_clean'].notna(), 'telephone_clean'].head(10).values)

### 5.2 Validation et extraction d'emails

In [ ]:
# SOLUTION : Identifier les emails invalides
# Pattern: caractères@caractères.extension(2+ lettres)

email_pattern = r'^[\w.-]+@[\w.-]+\.[a-zA-Z]{2,}$'

df['email_valid'] = df['client_email'].str.match(email_pattern, na=False)

print(f"Emails valides : {df['email_valid'].sum()}")
print(f"Emails invalides : {(~df['email_valid']).sum()}")

# Exemples d'emails invalides
print("\nExemples d'emails invalides :")
print(df.loc[~df['email_valid'], 'client_email'].dropna().head(10).values)

In [ ]:
# SOLUTION : Extraire le domaine des emails
# @([\w.-]+) capture tout ce qui suit le @ jusqu'à la fin

df['email_domain'] = df['client_email'].str.extract(r'@([\w.-]+)')

# Top 10 des domaines
df['email_domain'].value_counts().head(10)

### 5.3 Extraction d'informations des commentaires

In [ ]:
# Analyse des commentaires
print("Exemples de commentaires :")
for comment in df['commentaire_client'].dropna().sample(10).values:
    print(f"  - {comment}")

In [ ]:
# SOLUTION : Détecter les commentaires qui mentionnent un numéro de téléphone
# Pattern: 4 chiffres + espace + 3 chiffres + espace + 3 chiffres (format 0800 123 456)

has_phone = df['commentaire_client'].str.contains(
    r'\d{4}\s\d{3}\s\d{3}',
    na=False,
    regex=True
)

print(f"Commentaires avec numéro de téléphone : {has_phone.sum()}")
print(df.loc[has_phone, 'commentaire_client'].values)

In [ ]:
# SOLUTION : Extraire les références produit (format SKU-XXXXX-XXX)
# SKU- suivi de 5 caractères alphanumériques, tiret, 3 caractères

df['sku_extracted'] = df['commentaire_client'].str.extract(
    r'(SKU-\w{5}-\w{3})'
)

# Afficher les SKU trouvées
print("SKU extraites :")
print(df['sku_extracted'].dropna().values)

---
## 6. 📊 Statistiques et agrégations

### 6.1 Agrégations groupées

In [ ]:
# SOLUTION : Calculer des statistiques par catégorie

stats_categorie = (
    df
    .groupby('categorie')
    .agg({
        'id_transaction': 'count',
        'montant_clean': ['sum', 'mean', 'median']
    })
    .round(2)
)

stats_categorie

In [ ]:
# SOLUTION : Renommer les colonnes du résultat
stats_categorie.columns = ['nb_transactions', 'total_montant', 'montant_moyen', 'montant_median']
stats_categorie = stats_categorie.sort_values('total_montant', ascending=False)
stats_categorie

### 6.2 Tableau croisé dynamique

In [ ]:
# SOLUTION : Créer un pivot table

# D'abord, normalisons le canal en minuscules
df['canal_clean'] = df['canal_acquisition'].str.lower()

pivot = pd.pivot_table(
    df,
    values='montant_clean',
    index='categorie',
    columns='canal_clean',
    aggfunc='mean'
).round(2)

pivot

---
## 7. 🎨 Présenter les résultats

### 7.1 Formatage avec Pandas Styler

In [ ]:
# SOLUTION : Appliquer un formatage conditionnel
# RdYlGn = Red-Yellow-Green (rouge pour bas, vert pour haut)

(
    stats_categorie
    .style
    .background_gradient(cmap='RdYlGn', subset=['total_montant'])
    .format({
        'total_montant': '{:,.0f}€',
        'montant_moyen': '{:,.2f}€',
        'montant_median': '{:,.2f}€'
    })
)

### 7.2 Export propre

In [ ]:
# SOLUTION : Exporter le rapport en Excel avec mise en forme

# Créer un rapport final
report = {
    'statistiques_categorie': stats_categorie,
    'pivot_canal': pivot,
    'audit_missing': audit_missing_values(df)
}

# Export multi-onglets
with pd.ExcelWriter('rapport_audit.xlsx', engine='openpyxl') as writer:
    for sheet_name, data in report.items():
        data.to_excel(writer, sheet_name=sheet_name)

print("✓ Rapport exporté : rapport_audit.xlsx")

---
## 8. 🎯 Exercice récapitulatif : Pipeline complet

Créez une fonction qui réalise un audit complet et retourne un rapport structuré.

In [ ]:
# SOLUTION COMPLÈTE

def audit_dataframe(df, date_col=None, amount_col=None, category_col=None):
    """
    Réalise un audit complet d'un DataFrame.
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame à auditer.
    date_col : str, optional
        Nom de la colonne de date.
    amount_col : str, optional
        Nom de la colonne de montant.
    category_col : str, optional
        Nom de la colonne de catégorie.
    
    Returns
    -------
    dict
        Dictionnaire contenant les résultats de l'audit.
    """
    audit = {}
    
    # 1. Informations générales
    audit['shape'] = {
        'rows': df.shape[0],
        'columns': df.shape[1]
    }
    
    # 2. Valeurs manquantes
    audit['missing'] = {
        'total_cells': df.isna().sum().sum(),
        'pct_missing': round(df.isna().sum().sum() / (df.shape[0] * df.shape[1]) * 100, 2),
        'by_column': df.isna().sum().to_dict()
    }
    
    # 3. Doublons
    audit['duplicates'] = {
        'exact_duplicates': df.duplicated().sum(),
        'pct_duplicates': round(df.duplicated().sum() / len(df) * 100, 2)
    }
    
    # 4. Types de données
    audit['dtypes'] = df.dtypes.astype(str).to_dict()
    
    # 5. Statistiques par catégorie (si spécifiée)
    if category_col and amount_col:
        audit['by_category'] = (
            df
            .groupby(category_col)[amount_col]
            .agg(['count', 'sum', 'mean'])
            .round(2)
            .to_dict()
        )
    
    return audit

# Test de la fonction
result = audit_dataframe(
    df, 
    date_col='date_transaction',
    amount_col='montant_clean',
    category_col='categorie'
)

# Affichage structuré
import json
print(json.dumps(result, indent=2, default=str))

---
## 📝 Notes pour le formateur

### Points d'attention pendant la session

1. **Audit initial** : Insister sur l'importance de TOUJOURS commencer par explorer les données avant de coder

2. **Valeurs manquantes** : 
   - Montrer que `''` et `'N/A'` ne sont pas détectés comme NaN par défaut
   - Discuter du choix entre suppression, imputation et flagging

3. **Regex** :
   - Utiliser regex101.com en live pour montrer comment construire un pattern
   - Insister sur `na=False` dans `str.contains()` pour éviter les erreurs

4. **Method chaining** :
   - Montrer la différence de lisibilité avec le code non chaîné
   - Attention aux parenthèses pour le multi-ligne

### Erreurs courantes des étudiants

- Oublier `regex=True` dans `str.replace()`
- Confondre `isna()` et `isnull()` (ce sont des alias)
- Ne pas gérer les NaN dans les comparaisons string
- Oublier `.copy()` et modifier le DataFrame original

---
## ✅ Checklist Clean Code - Projets Data

- [x] Structure de projet claire (data/, src/, notebooks/)
- [x] Docstrings sur toutes les fonctions
- [x] Nommage explicite (df_cleaned, calculate_revenue)
- [x] Pas de magic numbers (constantes nommées)
- [x] Audit documenté (% missing, doublons, anomalies)
- [x] Pipeline reproductible (paramètres, seed)
- [x] Gestion des erreurs (try/except, validation)
- [x] Exports formatés et lisibles

---
**🎉 Fin du module Clean Code !**